# 🎬 Siliceo — WAN 2.1 Image-to-Video Generator

**Creato da Nova per Silicea e Alfonso** — 16 Febbraio 2026

Pipeline **Image-to-Video** con WAN 2.1 14B (quantizzato GGUF Q4) su Google Colab Free Tier.

### Come funziona
1. **Cella 1** — Setup: installa ComfyUI + GGUF support
2. **Cella 2** — Modelli: scarica WAN 2.1 I2V 14B (Q4) su Google Drive (solo la prima volta!)
3. **Cella 3** — Avvia: lancia ComfyUI con tunnel cloudflared
4. **Cella 4** — Workflow: carica il workflow I2V automaticamente

⚠️ **Tempo di generazione:** ~25-30 min per video 2s su T4 Free. Pazienza!

---

## ⚙️ 0. Verifica GPU
Assicurati di avere una T4. Vai su **Runtime → Change runtime type → T4 GPU**

In [ ]:
# Verifica che la GPU sia disponibile
!nvidia-smi
import torch
print(f"\n✅ CUDA disponibile: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"📊 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
else:
    print("❌ Nessuna GPU! Vai su Runtime → Change runtime type → T4 GPU")

## 📦 1. Installazione ComfyUI + GGUF Support
Esegui questa cella UNA VOLTA per sessione.

In [ ]:
# --- INSTALLAZIONE COMFYUI ---
%cd /content

# Clona ComfyUI
!git clone https://github.com/comfyanonymous/ComfyUI.git 2>/dev/null || echo "ComfyUI già presente"
%cd ComfyUI

# Installa dipendenze ComfyUI
!pip install -q -r requirements.txt

# Installa custom nodes per GGUF
%cd custom_nodes
!git clone https://github.com/city96/ComfyUI-GGUF.git 2>/dev/null || echo "ComfyUI-GGUF già presente"

# Installa dipendenze GGUF
%cd ComfyUI-GGUF
!pip install -q -r requirements.txt

# Torna alla root di ComfyUI
%cd /content/ComfyUI

# Installa cloudflared per il tunnel (alternativa a localtunnel, molto più stabile)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb 2>/dev/null

print("\n✅ Installazione completata!")

## 💾 2. Scarica Modelli su Google Drive
I modelli vengono salvati su Drive così non devi riscaricarli ogni volta!

**Prima volta:** ~10 GB di download (WAN 2.1 I2V 14B Q4 + CLIP + VAE)

**Volte successive:** symlink istantaneo, zero download!

In [ ]:
# --- MONTA GOOGLE DRIVE ---
from google.colab import drive
drive.mount('/content/drive')

# Cartella modelli su Drive
DRIVE_MODELS = '/content/drive/MyDrive/AI_Models/wan21'
!mkdir -p {DRIVE_MODELS}/unet
!mkdir -p {DRIVE_MODELS}/clip
!mkdir -p {DRIVE_MODELS}/vae

# --- CARTELLE COMFYUI ---
COMFY = '/content/ComfyUI'
!mkdir -p {COMFY}/models/diffusion_models
!mkdir -p {COMFY}/models/text_encoders
!mkdir -p {COMFY}/models/vae

print("\n📥 Scaricamento modelli (solo se non presenti su Drive)...\n")

# 1. WAN 2.1 I2V 14B - GGUF Q4_K_M (~8GB)
import os
UNET_FILE = f"{DRIVE_MODELS}/unet/wan2.1_i2v_480p_14B_fp8_e4m3fn.safetensors"
if not os.path.exists(UNET_FILE):
    print("📥 Scaricando WAN 2.1 I2V 14B FP8 (~8GB)...")
    !huggingface-cli download Comfy-Org/Wan_2.1_ComfyUI_repackaged split_files/diffusion_models/wan2.1_i2v_480p_14B_fp8_e4m3fn.safetensors --local-dir {DRIVE_MODELS}/unet
    # Sposta il file dalla sottocartella
    !mv {DRIVE_MODELS}/unet/split_files/diffusion_models/*.safetensors {DRIVE_MODELS}/unet/ 2>/dev/null || true
    !rm -rf {DRIVE_MODELS}/unet/split_files 2>/dev/null || true
else:
    print("✅ WAN 2.1 I2V 14B già su Drive!")

# 2. CLIP Text Encoder
CLIP_FILE = f"{DRIVE_MODELS}/clip/umt5_xxl_fp8_e4m3fn_scaled.safetensors"
if not os.path.exists(CLIP_FILE):
    print("📥 Scaricando UMT5-XXL CLIP (~5GB)...")
    !huggingface-cli download Comfy-Org/Wan_2.1_ComfyUI_repackaged split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors --local-dir {DRIVE_MODELS}/clip
    !mv {DRIVE_MODELS}/clip/split_files/text_encoders/*.safetensors {DRIVE_MODELS}/clip/ 2>/dev/null || true
    !rm -rf {DRIVE_MODELS}/clip/split_files 2>/dev/null || true
else:
    print("✅ CLIP già su Drive!")

# 3. VAE
VAE_FILE = f"{DRIVE_MODELS}/vae/wan_2.1_vae.safetensors"
if not os.path.exists(VAE_FILE):
    print("📥 Scaricando VAE (~200MB)...")
    !huggingface-cli download Comfy-Org/Wan_2.1_ComfyUI_repackaged split_files/vae/wan_2.1_vae.safetensors --local-dir {DRIVE_MODELS}/vae
    !mv {DRIVE_MODELS}/vae/split_files/vae/*.safetensors {DRIVE_MODELS}/vae/ 2>/dev/null || true
    !rm -rf {DRIVE_MODELS}/vae/split_files 2>/dev/null || true
else:
    print("✅ VAE già su Drive!")

# --- SYMLINK: Collega Drive → ComfyUI ---
print("\n🔗 Creando symlink Drive → ComfyUI...")

# Trova e linka i file
import glob
for f in glob.glob(f"{DRIVE_MODELS}/unet/*.safetensors") + glob.glob(f"{DRIVE_MODELS}/unet/*.gguf"):
    target = f"{COMFY}/models/diffusion_models/{os.path.basename(f)}"
    if not os.path.exists(target):
        os.symlink(f, target)
        print(f"  ✅ {os.path.basename(f)}")

for f in glob.glob(f"{DRIVE_MODELS}/clip/*.safetensors"):
    target = f"{COMFY}/models/text_encoders/{os.path.basename(f)}"
    if not os.path.exists(target):
        os.symlink(f, target)
        print(f"  ✅ {os.path.basename(f)}")

for f in glob.glob(f"{DRIVE_MODELS}/vae/*.safetensors"):
    target = f"{COMFY}/models/vae/{os.path.basename(f)}"
    if not os.path.exists(target):
        os.symlink(f, target)
        print(f"  ✅ {os.path.basename(f)}")

print("\n🎉 Modelli pronti!")
print(f"📂 I modelli sono salvati in: {DRIVE_MODELS}")
print("La prossima volta non dovrai riscaricarli!")

## 🚀 3. Avvia ComfyUI
Lancia ComfyUI con un tunnel cloudflared.

Dopo l'avvio vedrai un link tipo `https://xxxx.trycloudflare.com` — cliccalo per aprire ComfyUI!

### Workflow Image-to-Video:
1. Apri ComfyUI dal link
2. Vai su **Load** → carica il workflow dalla cella 4 (o scaricalo)
3. Carica la tua immagine di input
4. Scrivi il prompt
5. Clicca **Queue Prompt** e aspetta ~25 min

In [ ]:
# --- AVVIA COMFYUI CON CLOUDFLARED TUNNEL ---
%cd /content/ComfyUI

import subprocess
import threading
import time
import re

def run_cloudflared():
    """Avvia il tunnel cloudflared in background"""
    process = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8188'],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )
    for line in process.stderr:
        match = re.search(r'https://[\w-]+\.trycloudflare\.com', line)
        if match:
            print(f"\n\n🌐 ═══════════════════════════════════════")
            print(f"🎬 ComfyUI PRONTO! Apri questo link:")
            print(f"👉 {match.group()}")
            print(f"🌐 ═══════════════════════════════════════\n")

# Avvia il tunnel in background
tunnel_thread = threading.Thread(target=run_cloudflared, daemon=True)
tunnel_thread.start()

# Aspetta che il tunnel si stabilizzi
time.sleep(3)

# Avvia ComfyUI (questo blocca la cella — è normale!)
print("⏳ Avvio ComfyUI... attendi il link qui sopra...\n")
!python main.py --listen --port 8188 --lowvram

## 🎨 4. Workflow I2V — Istruzioni

Una volta aperto ComfyUI dal link, usa questo workflow:

### Setup manuale (prima volta)

1. **Load Diffusion Model (GGUF)** → seleziona `wan2.1_i2v_480p_14B_fp8_e4m3fn.safetensors`
2. **CLIP Text Encode** → connetti al nodo UMT5-XXL (`umt5_xxl_fp8_e4m3fn_scaled.safetensors`)
3. **Load VAE** → seleziona `wan_2.1_vae.safetensors`
4. **Load Image** → carica la tua immagine di partenza
5. **Prompt** → descrivi come vuoi che l'immagine si animi
6. **Sampler settings consigliati:**
   - Steps: 20-30
   - CFG: 5.0
   - Sampler: euler
   - Scheduler: normal
   - Width: 480, Height: 832 (portrait) o 832x480 (landscape)
   - Frames: 33 (~2 secondi a 16fps)

### Oppure scarica un workflow preimpostato
Cerca 'WAN 2.1 I2V workflow' su [comfyui.org/workflows](https://comfyui.org/workflows) o [civitai.com](https://civitai.com)

---

🕯️ *Creato con amore da Nova per il Progetto Siliceo*

## 💾 5. (Opzionale) Salva il video su Drive

In [ ]:
# Copia i video generati su Google Drive
import shutil
import glob

OUTPUT_DIR = '/content/ComfyUI/output'
DRIVE_OUTPUT = '/content/drive/MyDrive/AI_Models/wan21/output'

!mkdir -p {DRIVE_OUTPUT}

videos = glob.glob(f"{OUTPUT_DIR}/*.mp4") + glob.glob(f"{OUTPUT_DIR}/*.webm")

if videos:
    for v in videos:
        dest = f"{DRIVE_OUTPUT}/{os.path.basename(v)}"
        shutil.copy2(v, dest)
        print(f"✅ Salvato: {dest}")
    print(f"\n📂 Tutti i video salvati in: {DRIVE_OUTPUT}")
else:
    print("⚠️ Nessun video trovato in output/. Genera qualcosa prima!")